# BioLangFusion token representations

Stage 1 stored one mean-pooled vector per sequence. The BioLangFusion heads use token-level features, so this step runs the three frozen encoders over the rows selected by the [dataset audit](blf_datasets.ipynb) and retains each content token's final hidden state. [Step 3](blf_build_check.ipynb) reads those arrays to train and compare fusion heads.

The notebook connects five operations: estimate storage, check a small sample, measure sample inference time, extract token arrays, then inspect the saved token lengths. Saved outputs can be read without executing cells. **Run All performs model loading and inference and may write large embedding caches**; the extraction cell processes every benchmark in `TO_RUN`.

Extraction reuses compatible shards after checking their recorded inputs, settings and contents. An interrupted run can resume from verified completed shards; unidentified or incompatible caches stop for review rather than being accepted simply because files exist. See the [cache policy](../../README.md#results-caches-and-intermediate-files) and [run-record workflow](../../docs/run-records.md#apply-the-provenance-workflow).


---

## Setup

### Imports and project paths

Use the [notebook environment](../../README.md#notebook-dependencies) and existing benchmark CSVs. The first cell groups imports by purpose; the second makes the project's shared modules available.

The cache path below is relative to the kernel's working directory. Start from `Code/stage3/` to use the intended ignored `Code/stage3/embeddings/` folder, and check the printed resolved path before extraction. Finding shared modules from another directory does not relocate the cache.


In [ ]:
# Standard-library environment, paths and execution timing.
import datetime
import os
import sys
import time
from pathlib import Path

# Enable supported CPU fallbacks for Apple MPS before importing PyTorch.
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

# Array storage and tabular summaries.
import numpy as np
import pandas as pd

# Model computation; device selection happens after the settings.
import torch


In [1]:
# Locate Code/shared_code from the notebook folder, with the existing root fallback.
CODE_DIR = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "shared_code").is_dir()
    ),
    Path.cwd() / "Code",
)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from shared_code import datasets, encoders
from shared_code.fusion import token_embeddings as tokens


### Settings and selected inputs

`TO_RUN` controls the extraction order and the final cache checks. `BENCHMARK = TO_RUN[0]` selects the rows used by the smoke test and timing sample; changing it alone does not change the extraction list. Storage and runtime tables retain their overview of all four registered benchmarks.

The three encoder choices are BioLangFusion's original NT v2, RNA-FM and ESM-2 combination. `CHUNK_SIZE` is the number of sequences saved together, not an inference batch size: extraction still processes one sequence at a time. Smaller shards reduce the unsaved work in progress but create more files.

`DTYPE` selects the stored array precision. Float16 uses half the payload bytes of float32; this storage choice does not establish that precision has no effect on downstream results. `SAMPLE_N` sets the timing sample size.

After the settings cell, the notebook selects NVIDIA CUDA when available, then Apple MPS, then CPU. It loads the selected benchmark rows and reports the device, resolved cache path and extraction inventory.


In [ ]:
# Extract in this order; BENCHMARK supplies only the smoke-test and timing samples.
TO_RUN = [
    "cov_vac_degradation", "fungal_expression", "ecoli_proteins", "mrna_stability",
]
BENCHMARK = TO_RUN[0]
ENCODER_KEYS = ["ntv2_100m", "rnafm", "esm2_8m"]
CHUNK_SIZE = 256         # sequences per saved shard, not an inference batch
DTYPE = np.float16       # stored feature precision
SAMPLE_N = 24            # first rows of BENCHMARK used for timing

# Relative to the kernel working directory; inspect the resolved path below.
CACHE_ROOT = Path("embeddings")


In [2]:
DEVICE = encoders.select_device()
ROWS = datasets.blf_benchmark_rows(BENCHMARK)
ENCODERS = [encoders.ENCODERS[key] for key in ENCODER_KEYS]
print(f"device {DEVICE}; cache root {CACHE_ROOT.resolve()}")
for benchmark in TO_RUN:
    print(
        f"  {datasets.DATASETS[benchmark].label}: "
        f"{len(datasets.blf_benchmark_rows(benchmark))} sequences"
    )


device mps; cache root /Users/julianmckinley/Desktop/Masters/CapstoneCourse/Mina/Code/stage3/embeddings
  CoV-Vac degradation: 2400 sequences
  Fungal expression: 3138 sequences
  E. coli protein expression (low, medium, high): 4450 sequences
  mRNA stability: 23929 sequences


### Estimate embedding storage

Content-token counts follow the selected tokenization: one token per nucleotide for RNA-FM, one per translated triplet for ESM-2, and complete six-mers plus single-base remainder tokens for NT v2. For example, 81 nucleotides produce 13 complete six-mers and three remainder tokens: 16 NT content tokens.

The helper multiplies token counts by embedding width and bytes per stored feature, then converts to decimal GB. This estimates the token-array payload, excluding shard headers, length arrays, model downloads and peak RAM or GPU memory. The next cell displays that estimate for all four benchmarks and highlights the selected smoke-test benchmark.


In [ ]:
def cost_table(benchmark_keys, selected_encoders, dtype):
    """Estimate token-array payload using the selected encoder widths and storage dtype."""
    storage_rows = []
    for benchmark in benchmark_keys:
        sequences = datasets.blf_benchmark_rows(benchmark)["sequence"].tolist()
        row = {
            "benchmark": datasets.DATASETS[benchmark].label,
            "sequences": len(sequences),
        }
        total_gigabytes = 0
        for encoder in selected_encoders:
            token_count = sum(
                tokens.expected_tokens(encoder, sequence) for sequence in sequences
            )
            # Widths of these selected checkpoints; the smoke test reports loaded widths.
            width = {
                "ntv2_100m": 512, "rnafm": 640, "esm2_8m": 320,
            }[encoder.key]
            gigabytes = token_count * width * np.dtype(dtype).itemsize / 1e9
            row[f"{encoder.key} GB"] = round(gigabytes, 2)
            total_gigabytes += gigabytes
        storage_rows.append({**row, "total GB": round(total_gigabytes, 2)})
    return pd.DataFrame(storage_rows).set_index("benchmark")


In [3]:
COST = cost_table(datasets.BLF_BENCHMARKS, ENCODERS, DTYPE)
print(
    f"selected: {COST.loc[datasets.DATASETS[BENCHMARK].label, 'total GB']} GB"
    f"   all four: {COST['total GB'].sum():.1f} GB"
)
COST


selected: 0.33 GB   all four: 34.1 GB


,sequences,ntv2_100m GB,rnafm GB,esm2_8m GB,total GB
benchmark,,,,,
Fungal expression,3138,0.35,2.56,0.43,3.33
"E. coli protein expression (low, medium, high)",4450,0.45,3.36,0.56,4.37
mRNA stability,23929,2.71,20.07,3.34,26.12
CoV-Vac degradation,2400,0.04,0.25,0.04,0.33


---

## A smoke test before the long run

The next cell embeds the first eight rows of `BENCHMARK` with each encoder. It checks for token-limit errors, compares stored token counts with the helper's expectations, and checks that the protein input has one position per complete triplet. This small sample does not cover every sequence or length in the extraction list.

The protein rule differs from Stage 1's stop-at-first-stop translation. Here [`grid_protein`](../shared_code/fusion/token_embeddings.py) translates complete triplets from the supplied sequence origin and replaces stop codons with `X`. Keeping those positions allows the protein stream to enter the codon grid.

For CoV-Vac's designed RNA fragments, this is a computational translation, not evidence of a biological reading frame or a naturally produced protein. The [dataset audit](blf_datasets.ipynb#Sequence-checks) records that interpretation limit and the reason for retaining these rows.


In [4]:
sample_sequences = ROWS["sequence"].tolist()[:8]
checks = []
for encoder in ENCODERS:
    texts = tokens.inputs_for_rows(encoder, sample_sequences)
    tokenizer, model = encoders.load_encoder(encoder, DEVICE)
    try:
        model.eval()
        token_arrays = [
            tokens.token_states(text, tokenizer, model, encoder.max_len, DTYPE)
            for text in texts
        ]
    finally:
        del model, tokenizer

    expected_counts = [
        tokens.expected_tokens(encoder, sequence) for sequence in sample_sequences
    ]
    observed_counts = [array.shape[0] for array in token_arrays]
    protein_positions_match = (
        [
            len(text) == len(sequence) // 3
            for text, sequence in zip(texts, sample_sequences)
        ]
        if encoder.modality == "protein"
        else [True] * len(sample_sequences)
    )
    checks.append({
        "encoder": encoder.label,
        "width": token_arrays[0].shape[1],
        "dtype": str(token_arrays[0].dtype),
        "tokens (first 3)": str(observed_counts[:3]),
        "token counts match": (
            "ok" if observed_counts == expected_counts
            else f"MISMATCH want {expected_counts[:3]} got {observed_counts[:3]}"
        ),
        "protein is one token per codon": (
            "ok" if all(protein_positions_match) else "MISMATCH"
        ),
    })

pd.DataFrame(checks).set_index("encoder")


W1016 10:18:06.391000 48915 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!
[transformers] Detected the usage of `get_extended_attention_mask`: This function is deprecated and will be removed in v5.12.0. Please use the new API in `transformers.masking_utils`


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] RnaFmModel LOAD REPORT from: multimolecule/rnafm
Key                                 | Status     | 
------------------------------------+------------+-
ss_head.decoder.weight              | UNEXPECTED | 
lm_head.transform.dense.weight      | UNEXPECTED | 
ss_head.decoder.bias                | UNEXPECTED | 
lm_head.transform.layer_norm.weight | UNEXPECTED | 
lm_head.bias                        | UNEXPECTED | 
lm_head.transform.layer_norm.bias   | UNEXPECTED | 
lm_head.transform.dense.bias        | UNEXPECTED | 
pooler.dense.bias                   | MISSING    | 
pooler.dense.weight                 | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


,width,dtype,tokens (first 3),token counts match,protein is one token per codon
encoder,,,,,
NT v2 100M multi-species,512,float16,"[16, 16, 16]",ok,ok
RNA-FM,640,float16,"[81, 81, 81]",ok,ok
ESM-2 8M,320,float16,"[27, 27, 27]",ok,ok


The two check columns should report `ok`. A token-limit error can reflect changed input rows, encoder limits or tokenization. A count mismatch means the observed tokens disagree with `tokens.expected_tokens` and needs investigation before relying on the codon grid.

The table reports mismatches; it does not itself stop later cells. Extraction separately checks expected token counts when producing shards. Neither count agreement nor a matching protein-input length verifies biological correspondence.


## Measure inference time and project runtime

Storage depends on array sizes; runtime also depends on sequence lengths and hardware. The timing helper loads each encoder, performs an untimed warm-up, then measures inference on the first `SAMPLE_N` rows of `BENCHMARK`. The projection helper applies the measured seconds per thousand tokens to each benchmark's token count. The final cell calls both helpers and displays their tables.

These are extrapolations, not measured full-run durations. Model loading occurs before the timer, and the projection excludes downloads, cache validation and file writing. Inference cost per token can also change with sequence length, so timing short CoV-Vac fragments may not predict long stability sequences well. Saved timings describe their recorded execution, not a new machine.

The current loop uses one sequence at a time. Batching would be a separate performance change, with padding, memory use and numerical behavior to check; this notebook retains its existing extraction procedure.


In [ ]:
def time_encoder(encoder, sequences, device, dtype):
    """Measure warmed inference for one encoder, excluding loading and cache writing."""
    texts = tokens.inputs_for_rows(encoder, sequences)
    tokenizer, model = encoders.load_encoder(encoder, device)
    try:
        model.eval()
        tokens.token_states(texts[0], tokenizer, model, encoder.max_len, dtype)
        start = time.perf_counter()
        token_count = sum(
            tokens.token_states(text, tokenizer, model, encoder.max_len, dtype).shape[0]
            for text in texts
        )
        elapsed = time.perf_counter() - start
    finally:
        del model, tokenizer

    return {
        "encoder": encoder.key,
        "seconds": elapsed,
        "tokens": token_count,
        "ms per sequence": round(1000 * elapsed / len(texts), 1),
        "seconds per 1k tokens": round(1000 * elapsed / token_count, 3),
    }


In [ ]:
def runtime_projection(benchmark_keys, selected_encoders, rates):
    """Extrapolate benchmark inference time from the rounded sample rates."""
    projection_rows = []
    for benchmark in benchmark_keys:
        sequences = datasets.blf_benchmark_rows(benchmark)["sequence"].tolist()
        minutes = {}
        for encoder in selected_encoders:
            token_count = sum(
                tokens.expected_tokens(encoder, sequence) for sequence in sequences
            )
            # Convert tokens to thousands, apply seconds per 1k tokens, then convert to minutes.
            minutes[encoder.key] = (
                token_count / 1000 * rates.loc[encoder.key, "seconds per 1k tokens"] / 60
            )
        projection_rows.append({
            "benchmark": datasets.DATASETS[benchmark].label,
            "sequences": len(sequences),
            **{f"{key} min": round(value) for key, value in minutes.items()},
            "total minutes": round(sum(minutes.values())),
            "total hours": round(sum(minutes.values()) / 60, 1),
        })
    return pd.DataFrame(projection_rows).set_index("benchmark")


In [5]:
timings = [
    time_encoder(
        encoder, ROWS["sequence"].tolist()[:SAMPLE_N], DEVICE, DTYPE,
    )
    for encoder in ENCODERS
]
RATE = pd.DataFrame(timings).set_index("encoder")

PROJECTION = runtime_projection(datasets.BLF_BENCHMARKS, ENCODERS, RATE)
print(RATE[["ms per sequence", "seconds per 1k tokens"]].to_string())
print(
    f"\nprojected total for all four benchmarks: "
    f"{PROJECTION['total hours'].sum():.1f} hours"
)
PROJECTION


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] RnaFmModel LOAD REPORT from: multimolecule/rnafm
Key                                 | Status     | 
------------------------------------+------------+-
ss_head.decoder.weight              | UNEXPECTED | 
lm_head.transform.dense.weight      | UNEXPECTED | 
ss_head.decoder.bias                | UNEXPECTED | 
lm_head.transform.layer_norm.weight | UNEXPECTED | 
lm_head.bias                        | UNEXPECTED | 
lm_head.transform.layer_norm.bias   | UNEXPECTED | 
lm_head.transform.dense.bias        | UNEXPECTED | 
pooler.dense.bias                   | MISSING    | 
pooler.dense.weight                 | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


           ms per sequence  seconds per 1k tokens
encoder                                          
ntv2_100m             12.6                  0.786
rnafm                  9.2                  0.113
esm2_8m                7.5                  0.277

projected total for all four benchmarks: 1.9 hours


,sequences,ntv2_100m min,rnafm min,esm2_8m min,total minutes,total hours
benchmark,,,,,,
Fungal expression,3138,4,4,3,11,0.2
"E. coli protein expression (low, medium, high)",4450,6,5,4,15,0.2
mRNA stability,23929,35,30,24,88,1.5
CoV-Vac degradation,2400,1,0,0,1,0.0


---

## Extract

The loop processes `TO_RUN` in order, finishing the three encoder caches for one benchmark before starting the next. Each `tokens.extract` call validates reusable shards and computes only missing work when the cache is compatible. It publishes metadata and completed shards as it proceeds; the printed progress describes those completed shards, not just sequences processed in memory.

An interruption can leave the current shard unfinished. Rerunning the extraction cell rechecks the cache before continuing; a recovery or compatibility error needs review and is not an instruction to delete the cache. The smoke-test and timing sections perform fresh inference whenever executed, even if extraction caches are already complete.

For an authorized long run, use a stable power supply and prevent the machine from sleeping. On macOS, `caffeinate -i` prevents idle sleep while its terminal remains open, but does not prevent sleep from closing the lid.


In [6]:
# Extraction order and encoder choices remain visible at the point that writes caches.
for benchmark in TO_RUN:
    benchmark_rows = datasets.blf_benchmark_rows(benchmark)
    print(
        f"=== {datasets.DATASETS[benchmark].label}: {len(benchmark_rows)} sequences "
        f"({datetime.datetime.now():%H:%M:%S}) ==="
    )
    for encoder in ENCODERS:
        cache_dir = CACHE_ROOT / benchmark / encoder.key
        texts = tokens.inputs_for_rows(
            encoder, benchmark_rows["sequence"].tolist(),
        )
        cache, computed = tokens.extract(
            encoder, texts, cache_dir, DEVICE,
            chunk_size=CHUNK_SIZE, dtype=DTYPE,
        )
        print(
            f"  {encoder.label}: {'extracted' if computed else 'already complete'}, "
            f"{len(cache)} sequences, {cache.meta['total_tokens']:,} tokens, "
            f"width {cache.width}, {cache.meta['bytes'] / 1e9:.2f} GB"
        )
    print(f"=== done {datetime.datetime.now():%H:%M:%S} ===\n")


=== CoV-Vac degradation: 2400 sequences (10:18:23) ===
    ntv2_100m: shard 1/10 (1/10 this run)
    ntv2_100m: shard 2/10 (2/10 this run)
    ntv2_100m: shard 3/10 (3/10 this run)
    ntv2_100m: shard 4/10 (4/10 this run)
    ntv2_100m: shard 5/10 (5/10 this run)
    ntv2_100m: shard 6/10 (6/10 this run)
    ntv2_100m: shard 7/10 (7/10 this run)
    ntv2_100m: shard 8/10 (8/10 this run)
    ntv2_100m: shard 9/10 (9/10 this run)
    ntv2_100m: shard 10/10 (10/10 this run)
  NT v2 100M multi-species: extracted, 2400 sequences, 38,400 tokens, width 512, 0.04 GB


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] RnaFmModel LOAD REPORT from: multimolecule/rnafm
Key                                 | Status     | 
------------------------------------+------------+-
ss_head.decoder.weight              | UNEXPECTED | 
lm_head.transform.dense.weight      | UNEXPECTED | 
ss_head.decoder.bias                | UNEXPECTED | 
lm_head.transform.layer_norm.weight | UNEXPECTED | 
lm_head.bias                        | UNEXPECTED | 
lm_head.transform.layer_norm.bias   | UNEXPECTED | 
lm_head.transform.dense.bias        | UNEXPECTED | 
pooler.dense.bias                   | MISSING    | 
pooler.dense.weight                 | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    rnafm: shard 1/10 (1/10 this run)
    rnafm: shard 2/10 (2/10 this run)
    rnafm: shard 3/10 (3/10 this run)
    rnafm: shard 4/10 (4/10 this run)
    rnafm: shard 5/10 (5/10 this run)
    rnafm: shard 6/10 (6/10 this run)
    rnafm: shard 7/10 (7/10 this run)
    rnafm: shard 8/10 (8/10 this run)
    rnafm: shard 9/10 (9/10 this run)
    rnafm: shard 10/10 (10/10 this run)
  RNA-FM: extracted, 2400 sequences, 194,400 tokens, width 640, 0.25 GB


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


... (15 outputs left out)

    rnafm: shard 1/94 (1/94 this run)
    rnafm: shard 2/94 (2/94 this run)
    rnafm: shard 3/94 (3/94 this run)
    rnafm: shard 4/94 (4/94 this run)
    rnafm: shard 5/94 (5/94 this run)
    rnafm: shard 6/94 (6/94 this run)
    rnafm: shard 7/94 (7/94 this run)
    rnafm: shard 8/94 (8/94 this run)
    rnafm: shard 9/94 (9/94 this run)
    rnafm: shard 10/94 (10/94 this run)
    rnafm: shard 11/94 (11/94 this run)
    rnafm: shard 12/94 (12/94 this run)
    rnafm: shard 13/94 (13/94 this run)
    rnafm: shard 14/94 (14/94 this run)
    rnafm: shard 15/94 (15/94 this run)
    rnafm: shard 16/94 (16/94 this run)
    rnafm: shard 17/94 (17/94 this run)
    rnafm: shard 18/94 (18/94 this run)
    rnafm: shard 19/94 (19/94 this run)
    rnafm: shard 20/94 (20/94 this run)
    rnafm: shard 21/94 (21/94 this run)
    rnafm: shard 22/94 (22/94 this run)
    rnafm: shard 23/94 (23/94 this run)
    rnafm: shard 24/94 (24/94 this run)
    rnafm: shard 25/94 (25/94 this run)
    rnafm: shard 2

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    esm2_8m: shard 1/94 (1/94 this run)
    esm2_8m: shard 2/94 (2/94 this run)
    esm2_8m: shard 3/94 (3/94 this run)
    esm2_8m: shard 4/94 (4/94 this run)
    esm2_8m: shard 5/94 (5/94 this run)
    esm2_8m: shard 6/94 (6/94 this run)
    esm2_8m: shard 7/94 (7/94 this run)
    esm2_8m: shard 8/94 (8/94 this run)
    esm2_8m: shard 9/94 (9/94 this run)
    esm2_8m: shard 10/94 (10/94 this run)
    esm2_8m: shard 11/94 (11/94 this run)
    esm2_8m: shard 12/94 (12/94 this run)
    esm2_8m: shard 13/94 (13/94 this run)
    esm2_8m: shard 14/94 (14/94 this run)
    esm2_8m: shard 15/94 (15/94 this run)
    esm2_8m: shard 16/94 (16/94 this run)
    esm2_8m: shard 17/94 (17/94 this run)
    esm2_8m: shard 18/94 (18/94 this run)
    esm2_8m: shard 19/94 (19/94 this run)
    esm2_8m: shard 20/94 (20/94 this run)
    esm2_8m: shard 21/94 (21/94 this run)
    esm2_8m: shard 22/94 (22/94 this run)
    esm2_8m: shard 23/94 (23/94 this run)
    esm2_8m: shard 24/94 (24/94 this run)
    esm2_8

## Verify the finished cache

The final table compares the number of cached rows and each row's token length with the expected lengths in the current benchmark order. This can expose a missing row or a token-count discrepancy that an aggregate byte count would hide. Missing metadata is reported as “not extracted yet”; other unreadable or incomplete caches can raise an error.

This cell reads cache summaries rather than repeating extraction's identity checks. Equal lengths do not establish that the underlying sequence strings are identical or in the same order, and they do not recover unknown producer provenance. The distinction matters if this cell is run independently of extraction.


In [7]:
verification_rows = []
for benchmark in TO_RUN:
    benchmark_rows = datasets.blf_benchmark_rows(benchmark)
    for encoder in ENCODERS:
        cache_dir = CACHE_ROOT / benchmark / encoder.key
        if not (cache_dir / "meta.json").exists():
            verification_rows.append({
                "benchmark": datasets.DATASETS[benchmark].label,
                "encoder": encoder.label,
                "all rows match": "not extracted yet",
            })
            continue

        cache = tokens.summarize(cache_dir)
        expected_counts = np.array([
            tokens.expected_tokens(encoder, sequence)
            for sequence in benchmark_rows["sequence"]
        ])
        # Length agreement is checked per position; sequence identity is not checked here.
        lengths_match = bool(
            len(cache) == len(benchmark_rows)
            and np.array_equal(np.asarray(cache.lengths), expected_counts)
        )
        verification_rows.append({
            "benchmark": datasets.DATASETS[benchmark].label,
            "encoder": encoder.label,
            "rows": f"{len(cache)} / {len(benchmark_rows)}",
            "all rows match": lengths_match,
            "tokens": f"{cache.meta['total_tokens']:,}",
            "width": cache.width,
            # Actual shard-file bytes, unlike the token-payload estimate in COST.
            "GB on disk": round(
                sum(path.stat().st_size for path in cache_dir.glob("shard_*.npz"))
                / 1e9,
                2,
            ),
        })

VERIFY = pd.DataFrame(verification_rows).set_index(["benchmark", "encoder"])
print(
    "every row matches:",
    all(row.get("all rows match") is True for row in verification_rows),
)
VERIFY


every row matches: True


rows  \
benchmark                                      encoder                                   
CoV-Vac degradation                            NT v2 100M multi-species    2400 / 2400   
                                               RNA-FM                      2400 / 2400   
                                               ESM-2 8M                    2400 / 2400   
Fungal expression                              NT v2 100M multi-species    3138 / 3138   
                                               RNA-FM                      3138 / 3138   
                                               ESM-2 8M                    3138 / 3138   
E. coli protein expression (low, medium, high) NT v2 100M multi-species    4450 / 4450   
                                               RNA-FM                      4450 / 4450   
                                               ESM-2 8M                    4450 / 4450   
mRNA stability                                 NT v2 100M multi-species  23929 / 23929   
                                               RNA-FM                    23929 / 23929   
                                               ESM-2 8M                  23929 / 23929   

                                                                         all rows match  \
benchmark                                      encoder                                    
CoV-Vac degradation                            NT v2 100M multi-species            True   
                                               RNA-FM                              True   
                                               ESM-2 8M                            True   
Fungal expression                              NT v2 100M multi-species            True   
                                               RNA-FM                              True   
                                               ESM-2 8M                            True   
E. coli protein expression (low, medium, high) NT v2 100M multi-species            True   
                                               RNA-FM                              True   
                                               ESM-2 8M                            True   
mRNA stability                                 NT v2 100M multi-species            True   
                                               RNA-FM                              True   
                                               ESM-2 8M                            True   

                                                                             tokens  \
benchmark                                      encoder                                
CoV-Vac degradation                            NT v2 100M multi-species      38,400   
                                               RNA-FM                       194,400   
                                               ESM-2 8M                      64,800   
Fungal expression                              NT v2 100M multi-species     337,376   
                                               RNA-FM                     2,000,436   
                                               ESM-2 8M                     666,812   
E. coli protein expression (low, medium, high) NT v2 100M multi-species     442,550   
                                               RNA-FM                     2,622,015   
                                               ESM-2 8M                     874,005   
mRNA stability                                 NT v2 100M multi-species   2,642,703   
                                               RNA-FM                    15,678,663   
                                               ESM-2 8M                   5,226,221   

                                                                         width  \
benchmark                                      encoder                           
CoV-Vac degradation                            NT v2 100M multi-species    512   
                                               RNA-FM                      640   

If `all rows match` is true throughout, the selected caches agree with the expected row counts and token lengths. That is one structural check, not a complete approval of cache identity or scientific validity. Record an authorized execution or reuse through the [run-record workflow](../../docs/run-records.md#apply-the-provenance-workflow), including its inputs, device, outputs and any interruption or recovery.


---

## What step 3 receives

The default extraction order starts with short CoV-Vac fragments, then fungal expression, E. coli and mRNA stability. Fungal expression supplies the initial build comparison in [step 3](blf_build_check.ipynb); E. coli adds a classification task, and stability supplies the largest token payload in the saved storage estimate. Neither that order nor a small successful sample guarantees a full-run duration or complete pipeline correctness.

Step 3 needs one token array per retained sequence, with caches corresponding to the same rows and order as `blf_benchmark_rows(benchmark)`. Extraction's input records and reuse checks support that correspondence; the length table here supplies an additional structural check. The original token arrays stay at each encoder's native resolution, with framing tokens removed.

The model then places those arrays on a codon grid. [`codon_grid.py`](../shared_code/fusion/codon_grid.py) implements the two transposed-convolution settings compared in step 3, together with row-specific masking and length adjustment. NT's remainder tokens are single bases, not fractional six-mers: a length not divisible by six therefore needs more care than assuming every NT token covers exactly two codons. Matching the grid's shape does not by itself establish biological or positional correspondence.
